# M.S.A.F.E. — Dataset Pipeline + YOLOv8 Training (Colab)

Downloads the 3 Kaggle source datasets (chicken, pork, beef — Fresh/Spoiled), runs the
3-stage pipeline documented in `backend/dataset/README.md`, then trains and exports a
YOLOv8 classification model:

1. `prepare_dataset.py` — normalizes all sources into `dataset/raw/<species>/<class>/`
2. `build_annotated_split.py` — splits into train/val/test, YOLO classification format
3. `augment_dataset.py` — Albumentations on the training split only
4. Train YOLOv8-cls, evaluate on the held-out test set, export weights

**Before running:** get your Kaggle API key from [kaggle.com/settings](https://www.kaggle.com/settings) →
"Create New Token" — this downloads a `kaggle.json` file. You'll upload it in Step 2 below.
Never commit that file anywhere.

**This version saves training checkpoints to Google Drive** so a Colab disconnect doesn't
wipe your progress — see Step 5 below.

## Step 0: Get the repo

Colab starts with an empty runtime, so we need the actual pipeline scripts from the repo
rather than re-typing them here.

**If your repo is private**, replace the URL below with
`https://<YOUR_GITHUB_TOKEN>@github.com/Tr3sh4/GROUP-12-M.S.A.F.E.git` using a personal
access token (Settings → Developer settings → Personal access tokens on GitHub). Don't
leave your token pasted in a notebook you share or commit.

In [ ]:
!git clone https://github.com/Tr3sh4/GROUP-12-M.S.A.F.E.git
%cd GROUP-12-M.S.A.F.E/backend

## Step 1: Install the Kaggle CLI (and the pipeline's other dependencies)

In [ ]:
!pip install -q kaggle albumentations opencv-python-headless pillow numpy

## Step 2: Upload your Kaggle API key

Running this cell pops a file picker — choose the `kaggle.json` you downloaded from
kaggle.com/settings. It's copied into `~/.kaggle/` with the permissions Kaggle's CLI expects
and is *not* saved into the repo or this notebook.

**⚠️ Do not use Runtime → Run All past this point.** This cell needs you to manually pick a
file — "Run All" doesn't wait for that, so every cell after it silently runs with no Kaggle
credentials and no downloaded data, and you won't find out until training fails with a
confusing "no images found" error many steps later. Run each cell below one at a time with
the ▶ button (or Shift+Enter) instead.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your kaggle.json when prompted

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!rm kaggle.json  # don't leave a copy sitting in the Colab working directory

assert os.path.exists('/root/.kaggle/kaggle.json'), (
    "kaggle.json was not uploaded/copied correctly — re-run this cell and make sure "
    "you actually pick a file in the popup before it finishes. Everything after this "
    "point depends on this file existing."
)
print("kaggle.json is in place — safe to continue.")

## Step 3: Download the 3 datasets

See `dataset/README.md` for what each of these actually contains — note the pork one's
species isn't confirmed from its Kaggle listing, verify it once it's downloaded.

In [ ]:
!kaggle datasets download -d calvinsama/fresh-and-rotten-poultry-meat-datasets -p dataset/sources/chicken --unzip
!kaggle datasets download -d vinayakshanawad/meat-freshness-image-dataset -p dataset/sources/pork --unzip
!kaggle datasets download -d mexwell/locbeef-beef-quality-image-dataset -p dataset/sources/beef --unzip

In [ ]:
import os

for species in ("chicken", "pork", "beef"):
    path = f"dataset/sources/{species}"
    n = sum(len(files) for _, _, files in os.walk(path)) if os.path.exists(path) else 0
    print(f"dataset/sources/{species}: {n} files")
    assert n > 0, (
        f"dataset/sources/{species} is empty — the Kaggle download for {species} didn't "
        "work. Check the !kaggle datasets download output above for an auth error "
        "(most likely: kaggle.json missing/invalid — re-run Step 2)."
    )
print("All 3 sources downloaded something — safe to continue.")

## Step 4: Run the pipeline

Each script prints how many images it processed — if any stage reports 0 for a species,
the downloaded folder's class names didn't match what `prepare_dataset.py` expects; check
the printed warning (it names the folder to look at) and extend `CLASS_ALIASES` in that
script if needed.

In [ ]:
!python scripts/prepare_dataset.py

In [ ]:
import os

for species in ("chicken", "pork", "beef"):
    for cls in ("fresh", "spoiled"):
        path = f"dataset/raw/{species}/{cls}"
        n = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"dataset/raw/{species}/{cls}: {n}")

total = sum(len(files) for _, _, files in os.walk("dataset/raw")) if os.path.exists("dataset/raw") else 0
assert total > 0, (
    "dataset/raw is empty after prepare_dataset.py — scroll up to that cell's output for "
    "a '0 images' warning naming which species/folder it couldn't match, then check "
    "CLASS_ALIASES in scripts/prepare_dataset.py."
)
print(f"\n{total} total images in dataset/raw — safe to continue.")

## Step 4b: Merge in custom chicken/pork photos (not from Kaggle)

Your own labeled chicken/pork frames (extracted+labeled outside this notebook, see
`dataset/raw/<species>/<class>/*_fresh_*.jpg` / `*_spoiled_*.jpg` naming) don't come from
Kaggle, so `prepare_dataset.py` never sees them — they only exist as a zip in your Google
Drive. This cell merges them into `dataset/raw/` on top of what Step 4 just produced.

**Before running:** upload `custom_chicken_pork_frames.zip` to your Google Drive (any
folder), then set `ZIP_PATH` below to match where you put it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/custom_chicken_pork_frames.zip"  # <-- update if you put it elsewhere

assert os.path.exists(ZIP_PATH), (
    f"{ZIP_PATH} not found — upload custom_chicken_pork_frames.zip to your Google Drive "
    "first, then fix ZIP_PATH above to match where you put it."
)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall("dataset/raw")

print("Custom chicken/pork frames merged into dataset/raw/\n")
for species in ("chicken", "pork"):
    for cls in ("fresh", "spoiled"):
        path = f"dataset/raw/{species}/{cls}"
        n = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"dataset/raw/{species}/{cls}: {n} (Kaggle + custom combined)")

In [ ]:
!python scripts/build_annotated_split.py

In [ ]:
import os

total = 0
for split in ("train", "val", "test"):
    for cls in ("fresh", "spoiled"):
        path = f"dataset/annotated/{split}/{cls}"
        n = len(os.listdir(path)) if os.path.exists(path) else 0
        total += n
        print(f"dataset/annotated/{split}/{cls}: {n}")

assert total > 0, (
    "dataset/annotated is empty after build_annotated_split.py — this stage reads from "
    "dataset/raw, so if the previous cell's assertion passed but this one is empty, "
    "check build_annotated_split.py's own output above for errors."
)
print(f"\n{total} total images in dataset/annotated — safe to continue.")

In [ ]:
!python scripts/augment_dataset.py

In [ ]:
import os

total = 0
for cls in ("fresh", "spoiled"):
    path = f"dataset/augmented/train/{cls}"
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    total += n
    print(f"dataset/augmented/train/{cls}: {n}")

assert total > 0, (
    "dataset/augmented/train is empty after augment_dataset.py — check that cell's "
    "output above for errors (e.g. albumentations not installed correctly)."
)
print(f"\n{total} total images in dataset/augmented/train — safe to continue.")

## Sanity check

Quick per-split, per-class image counts — worth a glance before training on this, especially
to confirm the pork source didn't come back empty or wildly imbalanced against fresh/spoiled.

In [ ]:
import os

print("dataset/annotated (post-split, pre-augmentation):")
annotated_total = 0
for split in ("train", "val", "test"):
    for cls in ("fresh", "spoiled"):
        path = f"dataset/annotated/{split}/{cls}"
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        annotated_total += count
        print(f"  {split}/{cls}: {count}")

print("\ndataset/augmented/train (what YOLOv8 will actually train on):")
augmented_total = 0
for cls in ("fresh", "spoiled"):
    path = f"dataset/augmented/train/{cls}"
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    augmented_total += count
    print(f"  train/{cls}: {count}")

assert annotated_total > 0 and augmented_total > 0, (
    "One of the pipeline stages produced no images (see counts above) — do not "
    "proceed to training, it will fail with a confusing error. Re-run the "
    "earlier stage that shows 0."
)

---

## Step 5: Train YOLOv8

Everything above gets the data ready. From here on we actually train the model.

**Before running this section**: make sure Colab gave you a GPU — go to
**Runtime → Change runtime type → Hardware accelerator → T4 GPU** (or better, if you have
Colab Pro). Training on CPU works but is dramatically slower.

This is image *classification* (Fresh vs Spoiled for a whole photo), not object detection —
so we use Ultralytics' YOLOv8-cls variant, not the bounding-box YOLOv8 used for the "meat
freshness" Roboflow datasets we skipped earlier.

**Every restart/disconnect wipes Colab's local disk** — so this version mounts Google Drive
and saves all training checkpoints there (`project="/content/drive/MyDrive/msafe_runs"`).
If Colab disconnects mid-training, skip straight to the **Resume** cell near the bottom of
this section instead of restarting training from epoch 1 — as long as you've re-run
Steps 0–4 and the two cells directly below first (mount Drive, arrange data).

In [ ]:
!pip install -q ultralytics

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.")

### Mount Google Drive

Run this every session (fast if already mounted). Training checkpoints get saved here so a
disconnect doesn't lose your progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/msafe_runs', exist_ok=True)
print("Drive mounted and folder ready.")

### Arrange the data the way Ultralytics expects

Ultralytics classification training wants `train/`, `val/`, and `test/` as sibling folders
under one directory. Ours aren't siblings — `train` lives under `dataset/augmented/` (the
augmented version) while `val`/`test` live under `dataset/annotated/` (deliberately
*un*-augmented, per `augment_dataset.py`'s reasoning: evaluating on augmented data would be
grading the model on an easier, distorted version of the test).

Symlinks (not copies) tie them into one `dataset/yolo_data/` view without duplicating any
files on disk.

**Run this from the `backend/` folder** — if you get a "No images found" error later
pointing at `/content/dataset/yolo_data` (missing the repo folder name), run
`%cd /content/GROUP-12-M.S.A.F.E/backend` first, then re-run this cell.

In [ ]:
import os

yolo_data = os.path.abspath("dataset/yolo_data")
os.makedirs(yolo_data, exist_ok=True)

links = {
    "train": os.path.abspath("dataset/augmented/train"),
    "val": os.path.abspath("dataset/annotated/val"),
    "test": os.path.abspath("dataset/annotated/test"),
}
for split, target in links.items():
    assert os.path.exists(target) and os.listdir(target), (
        f"{target} is missing or empty — cannot link it as '{split}'. Go back and "
        "re-run the pipeline stage that should have populated it (see the sanity "
        "check cell above) before continuing."
    )
    link_path = os.path.join(yolo_data, split)
    if os.path.islink(link_path) or os.path.exists(link_path):
        os.remove(link_path) if os.path.islink(link_path) else None
    os.symlink(target, link_path)
    print(f"{split}: {link_path} -> {target}")

# Final check on the actual path Ultralytics will read from — this is what
# directly caused "No images found in dataset/yolo_data" during training.
for split in ("train", "val", "test"):
    n = sum(len(files) for _, _, files in os.walk(os.path.join(yolo_data, split)))
    assert n > 0, f"dataset/yolo_data/{split} resolves to 0 images — do not proceed to training."
    print(f"dataset/yolo_data/{split}: {n} images confirmed reachable")

### Train

`yolov8s-cls.pt` is the "small" YOLOv8 classification checkpoint (pretrained on ImageNet,
so it isn't starting from nothing) — roughly 3x the parameters of the `yolov8n-cls.pt`
nano checkpoint this notebook used originally, in exchange for meaningfully better
accuracy at some cost to training/inference speed. If accuracy still isn't good enough
after this, `yolov8m-cls.pt` is the next step up — but the real lever after this is more
representative training data (see `dataset/README.md`'s note on Philippine wet-market
conditions), not just a bigger checkpoint.

What each setting means:
- **epochs=50** — how many full passes over the training data. `patience=10` stops early if
  validation accuracy hasn't improved in 10 epochs, so this is a ceiling, not a fixed cost.
- **imgsz=224** — the standard input size for YOLOv8 classification (not 640, which is a
  detection-model size).
- **batch=16** — images processed per training step. Lowered from the nano checkpoint's 32
  since the small checkpoint uses more memory per image; raise it back toward 32 if Colab
  has headroom, or lower further (e.g. 8) if it reports a CUDA out-of-memory error.
- **patience=10** — early stopping, explained above. This is what "validation" is actually
  used for mechanically: deciding when training has stopped helping.
- **project="/content/drive/MyDrive/msafe_runs"** — saves every checkpoint to Google Drive
  instead of Colab's temporary local disk, so a disconnect doesn't erase your progress.

**Only run this cell to start a brand-new training run.** If you already have a run in
progress and got disconnected, use the **Resume** cell below instead — running this cell
again starts over from epoch 1.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s-cls.pt")

results = model.train(
    data=yolo_data,
    epochs=50,
    imgsz=224,
    batch=16,
    patience=10,
    project="/content/drive/MyDrive/msafe_runs",
    name="msafe_freshness",
)

### Resume — use this instead if Colab disconnected mid-training

After a disconnect: re-run Steps 0–4, then the two cells above (mount Drive, arrange data),
then run this cell instead of the "Train" cell above. It picks up from your last saved
checkpoint in Drive rather than starting over.

In [ ]:
import os

last_ckpt = "/content/drive/MyDrive/msafe_runs/msafe_freshness/weights/last.pt"
assert os.path.exists(last_ckpt), (
    f"No checkpoint found at {last_ckpt} — there's nothing to resume. "
    "Run the 'Train' cell above instead to start a fresh run."
)

from ultralytics import YOLO
model = YOLO(last_ckpt)
results = model.train(resume=True)

### Evaluate on the test set — do this exactly once

This is the "test" pile from `notes.txt` section 11: never touched until now. Run this
cell once, record the number, and don't go back and retrain-then-retest repeatedly chasing
a better test score — that defeats the point of keeping it held out (see the "why not just
train + test" explanation in that note).

In [ ]:
test_metrics = model.val(data=yolo_data, split="test")
print(f"Test top-1 accuracy: {test_metrics.top1:.4f}")
print(f"Test top-5 accuracy: {test_metrics.top5:.4f}")  # not meaningful with only 2 classes, printed for completeness

### (Optional) View training curves

Ultralytics saves a `results.png` per run with loss/accuracy curves — useful for spotting
overfitting (val accuracy flattening or dropping while train accuracy keeps climbing).

In [ ]:
from IPython.display import Image, display

display(Image(filename="/content/drive/MyDrive/msafe_runs/msafe_freshness/results.png"))

### Export the trained weights

Copies the best checkpoint (the one with the highest validation accuracy during training,
not necessarily the last epoch) to where the FastAPI backend expects to find it. Then
download `yolov8_msafe.pt` from Colab's file browser (left sidebar) onto your machine and
place it at `backend/app/models/yolov8_msafe.pt` locally.

**Don't `git add` it** — `.gitignore` deliberately excludes `app/models/*.pt` (large
binaries don't belong in git history); share/version trained weights as a release artifact
or a separate storage location instead.

In [ ]:
import shutil, os

os.makedirs("app/models", exist_ok=True)
best_weights = "/content/drive/MyDrive/msafe_runs/msafe_freshness/weights/best.pt"
shutil.copy(best_weights, "app/models/yolov8_msafe.pt")
print("Copied to app/models/yolov8_msafe.pt — download this file from Colab's file browser.")

---

## Next steps (not in this notebook)

The backend and mobile app are already wired up to use `yolov8_msafe.pt` — you don't need
to build any of that. All that's left:

1. Download `app/models/yolov8_msafe.pt` from Colab's file browser (left sidebar) and place
   it at `backend/app/models/yolov8_msafe.pt` on your machine, overwriting the empty
   placeholder that's there now. (It's also safely backed up in your Google Drive at
   `MyDrive/msafe_runs/msafe_freshness/weights/best.pt`.)
2. Make sure the backend is running (`uvicorn app.main:app --reload` from `backend/`) — it
   loads the model lazily on the first `/scan` request after the file is in place, so no
   restart is needed if it's already running.
3. Scan a real photo from the app and confirm you get back a real Fresh/Spoiled result
   instead of the "model weights not found" error.